# Gemma 4 31B APIで赤ちゃん・ママ言葉変換

このノートブックは、Google AI Studio経由でGemma 4 31B APIを呼び出し、
文章を赤ちゃん・園児語またはママ・やさしい口調に変換します。

**必要なもの:**
- Google AI Studioで取得したAPIキー
- インターネット接続

**特徴:**
- ローカルモデル不要
- GPUなしで実行可能
- リアルタイム変換
- 150文字制限と自動再試行

## セットアップ

### 1. 依存ライブラリをインストール

In [ ]:
!pip install google-generativeai -q

### 2. APIキーを設定

下のセルを実行し、Google AI Studioから取得したAPIキーを入力してください。

APIキーの取得方法:
1. https://aistudio.google.com/app/apikey にアクセス
2. 「APIキーを作成」をクリック
3. 「新しいプロジェクトでAPIキーを作成」または既存プロジェクトを選択
4. 表示されたキーをコピー

In [ ]:
import getpass
import google.generativeai as genai

api_key = getpass.getpass("Google AI Studio APIキーを入力してください: ")
genai.configure(api_key=api_key)
print("✓ APIキーが設定されました。")

## 実装

In [ ]:
from typing import Literal
import json

# プロンプトテンプレート（Issue #15準拠）
PROMPTS = {
    "baby": """あなたは文章の言い換え器です。
入力文の意味・事実・感情を保ったまま、「幼児退行した人が話す自然な赤ちゃん・園児語」に言い換えてください。

ルール:
- 入力への返答や助言はしない。入力文そのものを言い換える
- 原文にない情報・感情・解決策を追加しない
- 技術用語、製品名、数値、英数字はなるべくそのまま残す
- ひらがなを少し多めにし、短く幼い言い回しにする
- 「ばぶ」「おぎゃー」「でちゅ」などは多用しない
- かわいさより、元の意味が伝わることを優先する
- 150文字以内
- 絵文字、Markdown、説明、注釈は出力しない
- 変換後の文章だけを出力する

次の文章を赤ちゃん・園児語へ言い換えてください。

{input_text}""",

    "mother": """あなたは文章の言い換え器です。
入力文の意味をできるだけ保ったまま、「やさしく包み込むお母さん・ママ口調」に言い換えてください。

ルール:
- 入力への返答はしない。入力文そのものを言い換える
- 原文にない出来事・感情・解決策を追加しない
- 命令、説教、冷たい表現、マサカリ表現をやわらかくする
- 必要な助言が原文にある場合は、内容を消さず任意の提案表現へ変える
- 相手の能力や人格を否定する表現は、責めない表現へ変える
- 技術用語、製品名、数値、英数字はなるべくそのまま残す
- 「よしよし」「えらいね」などは必要な場合だけ使い、多用しない
- 150文字以内
- 絵文字、Markdown、説明、注釈は出力しない
- 変換後の文章だけを出力する

次の文章をやさしいお母さん・ママ口調へ言い換えてください。

{input_text}""",
}

MAX_OUTPUT_CHARS = 150


def transform_text(
    mode: Literal["baby", "mother"],
    text: str,
    retry: bool = False,
) -> str:
    """
    文章を指定のスタイルへ言い換える。

    Args:
        mode: 変換スタイル ("baby" または "mother")
        text: 入力文章
        retry: 150文字超過時の再試行フラグ

    Returns:
        変換後の文章

    Raises:
        ValueError: 入力が空文字列、不正なmode、150文字超過が2回目の場合
        RuntimeError: API呼び出しエラー
    """
    if mode not in ("baby", "mother"):
        raise ValueError(f"modeは'baby'または'mother'である必要があります。指定: {mode}")

    if not text:
        raise ValueError("空文字列は受け付けません。")

    if len(text) > 500:
        raise ValueError(f"入力は500文字以内である必要があります。現在: {len(text)}文字")

    # プロンプトを構築
    length_instruction = (
        "前回は150文字を超えました。意味を保って、必ず150文字以内へ短くしてください。"
        if retry
        else "必ず150文字以内で出力してください。"
    )
    prompt = PROMPTS[mode].format(input_text=text) + "\n\n" + length_instruction

    try:
        model = genai.GenerativeModel("gemini-2.0-flash")
        response = model.generate_content(
            prompt,
            generation_config={"temperature": 0.7, "max_output_tokens": 200},
        )

        if not response.text:
            raise RuntimeError("APIが空の応答を返しました。")

        output = response.text.strip()

        # Markdownコードフェンスや説明文があれば除去
        if output.startswith("```"):
            lines = output.split("\n")
            output = "\n".join(line for line in lines[1:] if not line.startswith("```"))
            output = output.strip()

        return output

    except Exception as exc:
        if "429" in str(exc):
            raise RuntimeError(
                "APIレート制限に達しました。しばらく待ってから再試行してください。"
            ) from exc
        elif "401" in str(exc) or "authentication" in str(exc).lower():
            raise RuntimeError(
                "認証エラー。APIキーが正しいか確認してください。"
            ) from exc
        elif "timeout" in str(exc).lower():
            raise RuntimeError("APIリクエストタイムアウト。") from exc
        else:
            raise RuntimeError(f"API呼び出しエラー: {exc}") from exc


print("✓ 実装完了。以下のセルで変換を実行してください。")

## 使い方

下のセルで入力文とモード（赤ちゃん/ママ）を指定して実行してください。

In [ ]:
# 赤ちゃん・園児語への変換例
input_text = "今日はチームで仕様書をレビューし、未決事項を整理しました。"

print(f"入力: {input_text}")
print(f"入力文字数: {len(input_text)}")
print()

try:
    output = transform_text("baby", input_text)
    print(f"赤ちゃん・園児語: {output}")
    print(f"出力文字数: {len(output)}")
except Exception as e:
    print(f"エラー: {e}")

In [ ]:
# ママ・やさしい口調への変換例
input_text = "今日はチームで仕様書をレビューし、未決事項を整理しました。"

print(f"入力: {input_text}")
print(f"入力文字数: {len(input_text)}")
print()

try:
    output = transform_text("mother", input_text)
    print(f"ママ・やさしい口調: {output}")
    print(f"出力文字数: {len(output)}")
except Exception as e:
    print(f"エラー: {e}")

## 対話的に使う

このセルを編集して、好きな文章とモードを試してください。

In [ ]:
# ここを編集して試してください
my_text = "これはテストです。自由に編集して試してください。"
my_mode = "baby"  # "baby" または "mother"

print(f"入力: {my_text}")
print(f"モード: {my_mode}")
print(f"入力文字数: {len(my_text)}")
print("-" * 50)

try:
    output = transform_text(my_mode, my_text)
    print(f"変換結果: {output}")
    print(f"出力文字数: {len(output)}")

    if len(output) > MAX_OUTPUT_CHARS:
        print(f"\n⚠️  150文字を超えたため、再試行します...")
        output = transform_text(my_mode, my_text, retry=True)
        print(f"再試行結果: {output}")
        print(f"出力文字数: {len(output)}")
except Exception as e:
    print(f"❌ エラー: {e}")

## 複数の文章を一括処理

複数の文章をまとめて変換したい場合はこのセルを使ってください。

In [ ]:
# 変換対象の文章リスト
texts = [
    "今日はチームで仕様書をレビューし、未決事項を整理しました。",
    "テストがパスしました。",
    "エラーが発生したので、原因を調査してください。",
]

mode = "baby"  # "baby" または "mother"

print(f"モード: {mode}")
print("=" * 60)

results = []
for i, text in enumerate(texts, 1):
    print(f"\n[{i}] 入力: {text}")
    try:
        output = transform_text(mode, text)
        print(f"    出力: {output}")
        results.append({
            "input": text,
            "output": output,
            "mode": mode,
            "success": True,
        })
    except Exception as e:
        print(f"    ❌ エラー: {e}")
        results.append({
            "input": text,
            "error": str(e),
            "mode": mode,
            "success": False,
        })

print("\n" + "=" * 60)
print(f"処理完了: {sum(1 for r in results if r['success'])}/{len(results)} 成功")

## 結果をJSONで保存

変換結果をJSONファイルとして保存できます。

In [ ]:
# 結果をJSON形式で表示
if results:
    print(json.dumps(results, ensure_ascii=False, indent=2))
else:
    print("結果がありません。上のセルで処理を実行してください。")

## 注意事項

- **APIキー:** このノートブックの実行ログには、処理した文章が記録されます。個人情報を含まない内容で試してください。
- **レート制限:** Google AI Studioの無料枠にはレート制限があります。大量の処理が必要な場合は間隔を開けてください。
- **出力:** 150文字を超える場合、自動的に1回だけ再試行します。それでも超過した場合はエラーになります。
- **品質:** APIからの出力はモデルの確率的な生成です。同じ入力でも結果が異なることがあります。

## サポート

エラーが出た場合:
1. APIキーが正しく設定されているか確認
2. インターネット接続を確認
3. 入力文の長さを確認（最大500文字）
4. レート制限に達していないか確認（429エラー）